# Cross-Validation and Model Optimization

This notebook evaluates the robustness and stability of QSAR models using cross-validation techniques and model optimization strategies.

The workflow includes:
- Hybrid molecular feature generation
- Cross-validation benchmarking
- Model stability analysis
- Hyperparameter optimization
- Optimized model comparison

# Cross-Validation of Hybrid Random Forest QSAR Model

This notebook evaluates the robustness and generalization performance of a Hybrid Random Forest QSAR model using cross-validation techniques.

The workflow includes:
- Hybrid molecular feature generation
- Cross-validation benchmarking
- Model stability analysis
- Performance evaluation
- Future hyperparameter optimization

In [2]:
import pandas as pd

df = pd.read_csv("final_12k_log_transformed_papp_dataset.csv")

In [3]:
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import Descriptors
from rdkit import RDLogger

import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor

from sklearn.model_selection import (
    cross_val_score,
    KFold
)

RDLogger.DisableLog('rdApp.*')

## Hybrid Molecular Feature Generation

Morgan fingerprints and physicochemical descriptors are combined to create a hybrid molecular representation for cross-validation analysis.

In [4]:
fingerprints = []
descriptor_list = []

for smi in df['canonical_smiles']:

    mol = Chem.MolFromSmiles(smi)

    if mol:

        # Morgan fingerprint
        fp = AllChem.GetMorganFingerprintAsBitVect(
            mol,
            radius=2,
            nBits=2048
        )

        fingerprints.append(np.array(fp))

        # Physicochemical descriptors
        descriptors = [
            Descriptors.MolWt(mol),
            Descriptors.MolLogP(mol),
            Descriptors.TPSA(mol),
            Descriptors.NumHDonors(mol),
            Descriptors.NumHAcceptors(mol),
            Descriptors.NumRotatableBonds(mol)
        ]

        descriptor_list.append(descriptors)

fingerprint_array = np.array(fingerprints)

descriptor_array = np.array(descriptor_list)

X_hybrid = np.hstack([
    fingerprint_array,
    descriptor_array
])

y = df['log_papp'].values

print(X_hybrid.shape)

(12290, 2054)


## 5-Fold Cross-Validation

5-fold cross-validation is performed to evaluate the robustness, stability, and generalization performance of the Hybrid Random Forest QSAR model.

The dataset is divided into five subsets, and the model is trained and evaluated across multiple train-test partitions.

In [5]:
rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

In [6]:
kfold = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [8]:
cv_scores = cross_val_score(
    rf_model,
    X_hybrid,
    y,
    cv=kfold,
    scoring='r2',
    n_jobs=1
)

In [9]:
print("Cross-validation R² scores:")

print(cv_scores)

print("\nMean R²:", cv_scores.mean())

print("Standard Deviation:", cv_scores.std())

Cross-validation R² scores:
[0.61048913 0.58218856 0.59707418 0.5962754  0.60377219]

Mean R²: 0.5979598904704191
Standard Deviation: 0.009414049457054301


# Hyperparameter Optimization of Hybrid Random Forest Model

RandomizedSearchCV is used to optimize the hyperparameters of the Hybrid Random Forest model using cross-validation.

The objective is to improve predictive performance while maintaining model stability and generalization capability.

In [10]:
from sklearn.model_selection import RandomizedSearchCV

In [11]:
param_grid = {

    'n_estimators': [100, 200, 300],

    'max_depth': [10, 20, 30, None],

    'min_samples_split': [2, 5, 10],

    'min_samples_leaf': [1, 2, 4],

    'max_features': ['sqrt', 'log2']
}

In [12]:
rf_model = RandomForestRegressor(
    random_state=42,
    n_jobs=-1
)

In [13]:
random_search = RandomizedSearchCV(

    estimator=rf_model,

    param_distributions=param_grid,

    n_iter=10,

    cv=5,

    scoring='r2',

    verbose=2,

    random_state=42,

    n_jobs=1
)

In [14]:
random_search.fit(X_hybrid, y)

Fitting 5 folds for each of 10 candidates, totalling 50 fits
[CV] END max_depth=None, max_features=log2, min_samples_leaf=2, min_samples_split=5, n_estimators=100; total time=   2.4s
[CV] END max_depth=None, max_features=log2, min_samples_leaf=2, min_samples_split=5, n_estimators=100; total time=   2.3s
[CV] END max_depth=None, max_features=log2, min_samples_leaf=2, min_samples_split=5, n_estimators=100; total time=   2.2s
[CV] END max_depth=None, max_features=log2, min_samples_leaf=2, min_samples_split=5, n_estimators=100; total time=   2.3s
[CV] END max_depth=None, max_features=log2, min_samples_leaf=2, min_samples_split=5, n_estimators=100; total time=   2.3s
[CV] END max_depth=None, max_features=log2, min_samples_leaf=4, min_samples_split=10, n_estimators=100; total time=   2.4s
[CV] END max_depth=None, max_features=log2, min_samples_leaf=4, min_samples_split=10, n_estimators=100; total time=   2.2s
[CV] END max_depth=None, max_features=log2, min_samples_leaf=4, min_samples_split=1

,estimator,RandomForestR...ndom_state=42)
,param_distributions,"{'max_depth': [10, 20, ...], 'max_features': ['sqrt', 'log2'], 'min_samples_leaf': [1, 2, ...], 'min_samples_split': [2, 5, ...], ...}"
,n_iter,10
,scoring,'r2'
,n_jobs,1
,refit,True
,cv,5
,verbose,2
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [15]:
print("Best Parameters:")

print(random_search.best_params_)

print("\nBest CV R²:")

print(random_search.best_score_)

Best Parameters:
{'n_estimators': 100, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'max_depth': None}

Best CV R²:
0.38071278513355605


# Final Conclusions: Hybrid Random Forest Cross-Validation and Hyperparameter Optimization

This study evaluated the robustness, stability, and optimization potential of a Hybrid Random Forest QSAR model for permeability prediction using cross-validation and hyperparameter tuning techniques.

The Hybrid Random Forest model utilized:
- Morgan fingerprint descriptors
- Physicochemical molecular descriptors

to capture both structural topology and global molecular behavior.

## Cross-Validation Analysis

### 1. Model Stability
5-fold cross-validation demonstrated highly stable model behavior across multiple dataset partitions.

Cross-validation results:

- Mean R² ≈ 0.598
- Standard Deviation ≈ 0.009

The low standard deviation indicates:
- strong generalization capability
- consistent predictive performance
- low sensitivity to dataset partitioning

These findings suggest that the Hybrid Random Forest model is robust and reliable for permeability prediction.

### 2. Generalization Performance
The cross-validation performance remained consistent with the original train-test benchmarking results, indicating that the model performance was not dependent on a single favorable data split.

This significantly increased confidence in:
- model reproducibility
- predictive stability
- applicability to unseen molecular data

## Hyperparameter Optimization

RandomizedSearchCV was applied to optimize the Hybrid Random Forest model using cross-validation-based parameter search.

The optimization explored parameters including:
- number of trees
- tree depth
- feature selection strategy
- split conditions
- leaf constraints

### 3. Optimization Outcome
The optimized model produced lower cross-validation performance compared to the baseline Hybrid Random Forest model.

This observation highlights an important machine learning principle:

- hyperparameter tuning does not always guarantee improved performance
- excessive constraints can reduce model flexibility
- sparse molecular fingerprint datasets may respond differently to parameter restrictions

The results suggest that the original Hybrid Random Forest configuration was already well-balanced for the current permeability dataset.

### 4. Scientific Interpretation
The tuning results demonstrated that:
- ensemble tree methods can already perform strongly with near-default settings
- aggressive parameter constraints may reduce learning capacity in sparse cheminformatics feature spaces
- model stability is as important as maximizing predictive metrics

## Overall Conclusion

The Hybrid Random Forest model demonstrated:
- strong predictive performance
- excellent cross-validation stability
- robust generalization behavior

The study further emphasized the importance of:
- cross-validation before optimization
- stability analysis
- careful interpretation of tuning outcomes
- scientifically reliable benchmarking practices

Overall, the baseline Hybrid Random Forest model remained a highly competitive and reliable QSAR approach for permeability prediction on the current dataset.

This workflow establishes a strong methodological foundation for future studies involving:
- Support Vector Regression optimization
- repeated cross-validation
- SHAP interpretability analysis
- applicability domain analysis
- scaffold-aware validation
- advanced molecular machine learning methods